In [1]:
# Load data (if not already loaded)
# Uncomment if needed:
import sys
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DataLoad").getOrCreate()
df_2018 = spark.read.format("parquet").load("0917_2017_18_with_2017_cost.parquet")
df_og = df_2018.toPandas()

import importlib
import pandas as pd
import numpy as np
import public.model_IAI

SEED = 123
BIN_FLAG_COLUMNS = public.model_IAI.get_bin_flag_columns(df_og) +['lab_monitoring_adherent','nephrology_consult_adherent','early_nephrology_referral']
STAGE_COLUMNS = [col for col in df_og.columns if "stage" in col.lower()]
#["stage_2017",'2017Q1_max_ckd_stage','2017Q2_max_ckd_stage', '2017Q3_max_ckd_stage','2017Q4_max_ckd_stage']
CAT_COLUMNS = df_og.select_dtypes(include=["object","category"]).columns.tolist()
TRUE_NUM_COLUMNS = public.model_IAI.get_true_num_columns(df_og,CAT_COLUMNS,BIN_FLAG_COLUMNS)+[ 'util_2017', 'total_increasing_quarters_2017'
, 'total_lab_tests', 'ckd_visit_count', 'quarters_with_labs', 'nephrology_visit_count', 'days_to_nephrology','MEDIAN_INCOME']
COST_COLUMNS = [col for col in df_og.columns if 
            "cost" in col.lower() or 
             "quarterly" in col.lower()  or "increasing" in col.lower()
             ]
UTILIZATION_COLUMNS = [col for col in df_og.columns if "claims" not in col.lower() ] + ['util_2017']
print("categorical cols: ", CAT_COLUMNS)
print("stage cols: ", STAGE_COLUMNS)
print(COST_COLUMNS)
leftover_cols = [
    c for c in df_og.columns 
    if c not in CAT_COLUMNS and c not in TRUE_NUM_COLUMNS and c not in STAGE_COLUMNS and c not in BIN_FLAG_COLUMNS 
]

print(f"Number of leftover columns: {len(leftover_cols)}")
print(leftover_cols, df_og.shape)  # preview first 50

def make_cost_stratum_3class(df):
    # Default to low-cost (class 0)
    cost_stratum = pd.Series(0, index=df.index)
    cost_stratum[(df['highcost_gt_50000'] == 1) & (df['highcost_gt_100000'] == 0)] = 1
    # Emergent high cost (class 2): 100k to 200k
    cost_stratum[(df['highcost_gt_100000'] == 1) & (df['highcost_gt_200000'] == 0)] = 2
    # High cost (class 2): 200k+
    cost_stratum[df['highcost_gt_200000'] == 1] = 3
    return cost_stratum

# Add the new column to your data
df_og['cost_stratum_2018'] = make_cost_stratum_3class(df_og)
print(df_og["cost_stratum_2018"].value_counts(dropna=False))

cutoff_columns = [col for col in df_og.columns if col.startswith('highcost_gt_')]

feature_cols = [c for c in df_og.columns
              if c not in  (['annual_cost_2017','annual_cost_2018_deflated',"ENROLID", "cost_stratum_2018"] 
              + cutoff_columns)]    # keep only predictors excl. cost of 2018 and cutoff of 2017
numeric_cols = df_og[feature_cols + ["cost_stratum_2018"]].select_dtypes(include=["number"]).columns
corrs = df_og[numeric_cols].corr()["cost_stratum_2018"].abs().sort_values(ascending=False)
# Columns to drop
high_corr_cols = corrs[corrs > 0.5].index.tolist()
# Remove the target column itself, if present
high_corr_cols = [col for col in high_corr_cols if col != "cost_stratum_2018"]
# Final filtered feature set
feature_cols = [col for col in feature_cols if col not in high_corr_cols]
print("High corr features dropped from prediction columns: ",high_corr_cols)
target_col = "highcost_gt_200000"

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/18 11:44:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/18 11:44:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/18 11:44:59 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/02/18 11:45:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


categorical cols:  ['ENROLID', 'INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION', 'cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity']
stage cols:  ['stage_2017', '2017Q1_max_ckd_stage', '2017Q2_max_ckd_stage', '2017Q3_max_ckd_stage', '2017Q4_max_ckd_stage']
['annual_cost_2017', 'highcost_gt_50000_2017', 'highcost_gt_75000_2017', 'highcost_gt_100000_2017', 'highcost_gt_200000_2017', 'highcost_gt_300000_2017', 'highcost_gt_400000_2017', 'highcost_gt_500000_2017', 'annual_cost_2018_deflated', 'highcost_gt_50000', 'highcost_gt_75000', 'highcost_gt_100000', 'highcost_gt_200000', 'highcost_gt_300000', 'highcost_gt_400000', 'highcost_gt_500000', '2017Q1_ckd_cost', '2017Q1_direct_ckd_cost', '2017Q1_procedure_ckd_cost', '2017Q1_comorbidity_ckd_cost', '2017Q2_ckd_cost', '2017Q2_direct_ckd_cost', '2017Q2_procedure_ckd_cost', '2017Q2_comorbidity_ckd_cost', '2017Q3_ckd_cost', '2017Q3_direct_ckd_cost', '2017Q3_procedure_ckd_cost', '2017Q3_comorbidity_ckd_cost', '2017Q4_ckd_cost', '2017Q4

In [2]:

train_ids, test_ids,train_pd,test_pd = public.model_IAI.train_test_split_enrol(df_og,
    target_col = "cost_stratum_2018",test_size=0.3,verbose=False,random_state=SEED)

print(f"Train shape: {train_pd.shape}, Test shape: {test_pd.shape}")
print("Feature cols:", len(feature_cols))

val_ids, test_ids, val_pd, test_pd = public.model_IAI.train_test_split_enrol(
    test_pd, 
    target_col=target_col,
    test_size=0.5,
    verbose=False, 
    random_state=SEED
)
X_test = test_pd[feature_cols]
y_test = test_pd[target_col]
X_val = val_pd[feature_cols]
y_val = val_pd[target_col]

print(f"Train: {train_pd.shape}, Val: {val_pd.shape}, Test: {test_pd.shape}")
print(f"Train target distribution:\n{train_pd[target_col].value_counts()}")


Train shape: (23479, 96), Test shape: (10063, 96)
Feature cols: 78
Train: (23479, 96), Val: (5031, 96), Test: (5032, 96)
Train target distribution:
highcost_gt_200000
0    22819
1      660
Name: count, dtype: int64


In [3]:
# DIAGNOSTIC: Check for object dtype columns in feature_cols
print("="*80)
print("DIAGNOSTIC: Checking for problematic columns")
print("="*80)

print(f"\nTotal feature_cols: {len(feature_cols)}")
print(f"CAT_COLUMNS in feature_cols: {len([c for c in CAT_COLUMNS if c in feature_cols])}")
print(f"TRUE_NUM_COLUMNS in feature_cols: {len([c for c in TRUE_NUM_COLUMNS if c in feature_cols])}")

# Check dtypes of all feature columns
dtypes_summary = train_pd[feature_cols].dtypes.value_counts()
print(f"\nDtype distribution in feature_cols:")
for dtype, count in dtypes_summary.items():
    print(f"  {dtype}: {count} columns")

# Find object columns
object_cols = train_pd[feature_cols].select_dtypes(include=['object']).columns.tolist()
if object_cols:
    print(f"\n⚠ Found {len(object_cols)} object-type columns in feature_cols:")
    for col in object_cols[:20]:  # Show first 20
        unique_vals = train_pd[col].nunique()
        sample_val = train_pd[col].iloc[0]
        print(f"  - {col}: {unique_vals} unique values, sample: {sample_val}")
    
    # Check if they're in CAT_COLUMNS or TRUE_NUM_COLUMNS
    print(f"\n  Checking categorization:")
    for col in object_cols:
        in_cat = col in CAT_COLUMNS
        in_num = col in TRUE_NUM_COLUMNS
        print(f"    {col}: CAT={in_cat}, NUM={in_num}")
else:
    print(f"\n✓ No object-type columns found in feature_cols")

# Check for columns not in either list
accounted = set([c for c in CAT_COLUMNS if c in feature_cols] + 
                [c for c in TRUE_NUM_COLUMNS if c in feature_cols] + [c for c in BIN_FLAG_COLUMNS if c in feature_cols])
unaccounted = set(feature_cols) - accounted

if unaccounted:
    print(f"\n⚠ {len(unaccounted)} feature_cols not in CAT_COLUMNS or TRUE_NUM_COLUMNS or BIN_FLAG_COLUMNS:")
    for col in list(unaccounted)[:20]:
        dtype = train_pd[col].dtype
        print(f"  - {col}: {dtype}")
else:
    print(f"\n✓ All feature_cols are in CAT_COLUMNS or TRUE_NUM_COLUMNS")

print("\n" + "="*80)

# Check ENROLID dtype (this might be the actual issue!)
print("\n🔍 Checking ENROLID column dtype:")
print(f"  train_pd['ENROLID'].dtype: {train_pd['ENROLID'].dtype}")
if train_pd['ENROLID'].dtype == 'object':
    print("  ⚠ ENROLID is object dtype - this will cause HDF5 error!")
    print("  Sample values:", train_pd['ENROLID'].head(3).tolist())
else:
    print("  ✓ ENROLID is numeric - should be fine for HDF5")

print("="*80)


DIAGNOSTIC: Checking for problematic columns

Total feature_cols: 78
CAT_COLUMNS in feature_cols: 7
TRUE_NUM_COLUMNS in feature_cols: 46

Dtype distribution in feature_cols:
  int32: 35 columns
  float64: 29 columns
  object: 7 columns
  int64: 7 columns

⚠ Found 7 object-type columns in feature_cols:
  - INCOME_LEVEL: 4 unique values, sample: 0
  - AGEGRP: 5 unique values, sample: 5
  - SEX: 2 unique values, sample: 2
  - REGION: 5 unique values, sample: 3
  - cost_pattern_2017: 3 unique values, sample: late_heavy
  - cost_stability_2017: 3 unique values, sample: moderate
  - lab_monitoring_intensity: 4 unique values, sample: High

  Checking categorization:
    INCOME_LEVEL: CAT=True, NUM=False
    AGEGRP: CAT=True, NUM=False
    SEX: CAT=True, NUM=False
    REGION: CAT=True, NUM=False
    cost_pattern_2017: CAT=True, NUM=False
    cost_stability_2017: CAT=True, NUM=False
    lab_monitoring_intensity: CAT=True, NUM=False

✓ All feature_cols are in CAT_COLUMNS or TRUE_NUM_COLUMNS


🔍 

In [5]:
# PRECOMPUTE ALL PAIRWISE DISTANCES USING precompute_distances.py
import public.precompute_distances
importlib.reload(public.precompute_distances)
from public.precompute_distances import (
    compute_distances_batched,
    save_distances_hdf5,
    save_distances_numpy_memmap,
    precompute_leaf_dnn_memmap
)
import h5py
import time
import os
import json

print("="*80)
print("PRECOMPUTING PAIRWISE DISTANCES")
print("="*80)

# 1. Separate majority and minority classes
print("\n1. Separating data by class...")
majority_df = train_pd[train_pd[target_col] == 0].copy()
minority_df = train_pd[train_pd[target_col] == 1].copy()

print(f"  Majority (class 0): {len(majority_df):,} samples")
print(f"  Minority (class 1): {len(minority_df):,} samples")
print(f"  Total distances to compute: {len(majority_df) * len(minority_df):,}")

# Store ENROLIDs and ensure they're numeric for HDF5
print("  Converting ENROLIDs to numeric format...")
if majority_df['ENROLID'].dtype == 'object':
    # Convert object/string ENROLIDs to int64
    majority_enrolids = majority_df['ENROLID'].astype('int64').values
    minority_enrolids = minority_df['ENROLID'].astype('int64').values
    print(f"    ✓ Converted object ENROLIDs to int64")
else:
    majority_enrolids = majority_df['ENROLID'].values
    minority_enrolids = minority_df['ENROLID'].values
    print(f"    ✓ ENROLIDs already numeric: {majority_enrolids.dtype}")

# 2. Preprocess features using PushPullSampler logic
print("\n2. Preprocessing features (matching PushPullSampler)...")
from public.model_IAI import get_preprocessor_with_impute
from sklearn.impute import SimpleImputer

exclude_cols = ['ENROLID', target_col,"cost_stratum_2018","leaf_assignment", "predicted_cost_stratum"]+ [col for col in train_pd.columns.tolist() if "highcost_gt_" in col.lower()]
all_cols = feature_cols #[c for c in train_pd.columns if c not in exclude_cols]
print(f" Feature cols are the same as all_cols",feature_cols == all_cols)

# Combine majority and minority for consistent preprocessing
combined_df = pd.concat([
    majority_df[all_cols], 
    minority_df[all_cols]
], ignore_index=True)

# Categorize features (matching PushPullSampler logic)
numeric_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = combined_df.select_dtypes(include=["object", "category"]).columns.tolist()

# Build preprocessor (matching PushPullSampler)
preprocessor = get_preprocessor_with_impute(
    X_train=combined_df,
    categorical_cols=categorical_cols,
    numeric_cols=numeric_cols,
    binary_cols=BIN_FLAG_COLUMNS,
    verbose=True
)
# Transform
X_combined = preprocessor.fit_transform(combined_df)

# Split back into majority and minority
n_majority = len(majority_df)
X_majority = X_combined[:n_majority]
X_minority = X_combined[n_majority:]

print(f"  Preprocessed majority shape: {X_majority.shape}")
print(f"  Preprocessed minority shape: {X_minority.shape}")



PRECOMPUTING PAIRWISE DISTANCES

1. Separating data by class...
  Majority (class 0): 22,819 samples
  Minority (class 1): 660 samples
  Total distances to compute: 15,060,540
  Converting ENROLIDs to numeric format...
    ✓ Converted object ENROLIDs to int64

2. Preprocessing features (matching PushPullSampler)...
 Feature cols are the same as all_cols True
→ Building preprocessor w/ conditional imputation:
   • Cat: OHE on: ['INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION', 'cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity']
   • Num: scale on: ['has_Hypertension', 'has_Type_2_Diabetes', 'has_Anemia', 'has_Hyperlipidemia', 'has_Acute_Kidney_Failure', 'has_Hyperparathyroidism', 'has_Kidney_Transplant', 'has_Vitamin_D_Deficiency', 'has_Long-term_Drug_Therapy', 'has_Hypothyroidism', 'has_Sleep_Apnea', 'stage_2017', 'util_2017', 'Antihyperlipidemic Drugs, NEC (THRCLS_53)', 'Cardiac, Beta Blockers (THRCLS_51)', 'Cardiac, Calcium Channel (THRCLS_52)', 'Psychother, Antidepressa

In [ ]:

# VALIDATION: Ensure no object dtypes remain
print("\n  Validating preprocessed data...")
if hasattr(X_majority, 'dtype'):
    # NumPy array
    if X_majority.dtype == 'object':
        raise TypeError(f"Preprocessed X_majority still has object dtype!")
    print(f"    ✓ X_majority dtype: {X_majority.dtype}")
    print(f"    ✓ X_minority dtype: {X_minority.dtype}")
else:
    # Might be sparse or other format
    try:
        X_majority_dense = X_majority.toarray() if hasattr(X_majority, 'toarray') else X_majority
        X_minority_dense = X_minority.toarray() if hasattr(X_minority, 'toarray') else X_minority
        print(f"    ✓ Converted to dense arrays")
        print(f"    ✓ X_majority dtype: {X_majority_dense.dtype}")
        print(f"    ✓ X_minority dtype: {X_minority_dense.dtype}")
        # Update references
        X_majority = X_majority_dense
        X_minority = X_minority_dense
    except Exception as e:
        print(f"    ⚠ Could not validate dtype: {e}")

# Additional check: ensure numeric
try:
    _ = X_majority.astype(np.float32)
    _ = X_minority.astype(np.float32)
    print(f"    ✓ Data is numeric (can be cast to float32)")
except (ValueError, TypeError) as e:
    print(f"    ✗ ERROR: Data contains non-numeric values!")
    print(f"    {e}")
    raise

# 3. Compute distances
print("\n3. Computing pairwise distances...")
print(f"  Shape: {X_majority.shape} (majority) × {X_minority.shape} (minority)")
print(f"  Total distances: {X_majority.shape[0] * X_minority.shape[0]:,}")

start_time = time.time()

# Check if tqdm is available
try:
    from tqdm import tqdm
    has_tqdm = True
    print("  ✓ Using tqdm for progress bar")
except ImportError:
    has_tqdm = False
    print("  ⚠ tqdm not available, no progress bar")

# Compute distances with explicit batching and progress
batch_size = 1000
n_majority = X_majority.shape[0]
n_minority = X_minority.shape[0]

# Pre-allocate
distances = np.zeros((n_majority, n_minority), dtype=np.float32)
print(f"\n  Allocated {distances.nbytes / 1e6:.1f} MB for distance matrix")

# Compute in batches
n_batches = (n_majority + batch_size - 1) // batch_size
print(f"  Computing in {n_batches} batches of {batch_size} samples...")

if has_tqdm:
    batch_iterator = tqdm(range(n_batches), desc="Computing distances")
else:
    batch_iterator = range(n_batches)
    print("  Progress: ", end="", flush=True)

for i in batch_iterator:
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, n_majority)
    
    # Compute distances for this batch
    from sklearn.metrics import pairwise_distances
    batch_distances = pairwise_distances(
        X_majority[start_idx:end_idx], 
        X_minority,
        metric='euclidean'
    )
    
    distances[start_idx:end_idx] = batch_distances.astype(np.float32)
    
    # Print progress every 10 batches if no tqdm
    if not has_tqdm and i % 10 == 0:
        print(f"{i}/{n_batches}", end=" ", flush=True)

if not has_tqdm:
    print()  # New line after progress

elapsed = time.time() - start_time
print(f"\n  ✓ Completed in {elapsed/60:.1f} minutes ({elapsed:.1f} seconds)")
print(f"  ✓ Distance range: [{distances.min():.3f}, {distances.max():.3f}]")
print(f"  ✓ Distance mean: {distances.mean():.3f}, std: {distances.std():.3f}")
print(f"  ✓ Matrix shape: {distances.shape}")
print(f"  ✓ Matrix size: {distances.nbytes / 1e6:.1f} MB")

# 4. Save in multiple formats
output_dir = "./CKD_precomputed_distances_all_features"
os.makedirs(output_dir, exist_ok=True)

print(f"\n4. Saving results to {output_dir}/...")

# Save HDF5 (recommended)
print("\n  a) Saving HDF5 format...")
save_distances_hdf5(
    distances,
    majority_enrolids,
    minority_enrolids,
    f"{output_dir}/distances_majority_minority_seed_{SEED}.h5",
    compression='gzip'
)

"""print("\n  b) Saving Numpy format...")
#save_distances_numpy_memmap(
 #   distances,
 #   majority_enrolids,
 #   minority_enrolids,
 #   f"{output_dir}/distances_seed_{SEED}"
#)

# Save summary statistics
print("\n  c) Saving summary statistics...")
summary = {
    'n_majority': int(len(majority_enrolids)),
    'n_minority': int(len(minority_enrolids)),
    'n_distances': int(len(majority_enrolids) * len(minority_enrolids)),
    'n_features': int(X_majority.shape[1]),
    'distance_min': float(distances.min()),
    'distance_max': float(distances.max()),
    'distance_mean': float(distances.mean()),
    'distance_std': float(distances.std()),
    'distance_median': float(np.median(distances)),
    'dtype': str(distances.dtype),
    'compute_time_minutes': elapsed / 60,
    'batch_size': 1000,
    'preprocessing': {
        'n_cat_features': len(categorical_cols),
        'n_num_features': len(num_feats),
        'n_bin_features': len(bin_feats),
        'total_features_input': len(all_cols),
        'total_features_output': int(X_majority.shape[1])
    }
}

with open(f"{output_dir}/distance_summary_seed_{SEED}.json", 'w') as f:
    json.dump(summary, f, indent=2)"""

# Display file sizes
print("\n" + "="*80)
print("PRECOMPUTATION COMPLETE!")
print("="*80)
print(f"\nFiles saved to: {output_dir}/")
print(f"  - distances_majority_minority.h5")
print(f"      Size: {os.path.getsize(f'{output_dir}/distances_majority_minority_seed_{SEED}.h5') / 1e6:.1f} MB")

print(f"\nTotal computation time: {elapsed/60:.1f} minutes")
print(f"Average time per distance: {elapsed / (len(majority_enrolids) * len(minority_enrolids)) * 1e6:.2f} microseconds")
print("="*80)



  Validating preprocessed data...
    ✓ X_majority dtype: float64
    ✓ X_minority dtype: float64
    ✓ Data is numeric (can be cast to float32)

3. Computing pairwise distances...
  Shape: (22819, 122) (majority) × (660, 122) (minority)
  Total distances: 15,060,540
  ✓ Using tqdm for progress bar

  Allocated 60.2 MB for distance matrix
  Computing in 23 batches of 1000 samples...


Computing distances: 100%|██████████| 23/23 [00:00<00:00, 386.90it/s]


  ✓ Completed in 0.0 minutes (0.1 seconds)
  ✓ Distance range: [2.482, 192.007]
  ✓ Distance mean: 17.035, std: 12.071
  ✓ Matrix shape: (22819, 660)
  ✓ Matrix size: 60.2 MB

4. Saving results to ./CKD_precomputed_distances_all_features/...

  a) Saving HDF5 format...

Saving to HDF5: ./CKD_precomputed_distances_all_features/distances_majority_minority_seed_123.h5


  ✓ Saved 52.4 MB

PRECOMPUTATION COMPLETE!

Files saved to: ./CKD_precomputed_distances_all_features/
  - distances_majority_minority.h5
      Size: 52.4 MB

Total computation time: 0.0 minutes
Average time per distance: 0.00 microseconds

IMPORTANT: Preprocessing Details
The distances were computed using PushPullSampler's preprocessing:
  1. Excluded: ENROLID, target, cost columns
  2. Imputed: median (numeric), most_frequent (categorical)
  3. Categorized: categorical → one-hot, numeric → standardized, binary → unchanged
  4. This matches get_preprocessed_control_case_features() exactly

✓ These distances can be directly reused in PushPull optimization!


## PRECOMPUTE MAJORITY-TO-MAJORITY DISTANCES


In [7]:
output_dir = "./CKD_precomputed_distances_all_features"
os.makedirs(output_dir, exist_ok=True)
dnn_matrix_npy, dnn_enrolids_npy = precompute_leaf_dnn_memmap(
        X_majority_leaf=X_majority,
        majority_enrolids_leaf=majority_enrolids,
        out_dir=output_dir,
        leaf_id="global",  # Use "global" as leaf_id
        batch_size=750,
    )

[leaf global] computing d_nn for n=22,819 -> ~2.08 GB float32 (metric=euclidean)


leaf global d_nn: 100%|██████████| 31/31 [00:01<00:00, 24.29it/s]


[leaf global] saved:
  ./CKD_precomputed_distances_all_features/leaf_global_dnn_matrix.npy
  ./CKD_precomputed_distances_all_features/leaf_global_dnn_enrolids.npy


In [15]:
print("="*80)
print("PRECOMPUTING MAJORITY-TO-MAJORITY DISTANCES")
print("="*80)

print("\n1. Estimating computation size...")
n_majority = len(majority_df)
total_distances = n_majority * n_majority
unique_distances = n_majority * (n_majority - 1) // 2  # Upper triangle only

print(f"  Majority samples: {n_majority:,}")
print(f"  Total distances (full matrix): {total_distances:,}")
print(f"  Unique distances (upper triangle): {unique_distances:,}")
print(f"  Memory (float32): {total_distances * 4 / 1e9:.2f} GB")

# Time estimation based on previous computation
prev_time = 0.05  # seconds for 15M distances
prev_count = 15_060_540
estimated_time = (total_distances / prev_count) * prev_time
print(f"\n  ⏱ Estimated time: {estimated_time:.1f} seconds ({estimated_time/60:.2f} minutes)")

print("\n2. Computing pairwise distances...")
print(f"  Shape: {X_majority.shape} × {X_majority.shape}")
print(f"  Using batched computation for memory efficiency...")

start_time = time.time()

# Compute in batches to manage memory
batch_size = 1000
n_batches = (n_majority + batch_size - 1) // batch_size

# Pre-allocate distance matrix
distances_majority = np.zeros((n_majority, n_majority), dtype=np.float32)
print(f"\n  Allocated {distances_majority.nbytes / 1e9:.2f} GB for distance matrix")
print(f"  Computing in {n_batches} batches of {batch_size} samples...")

if has_tqdm:
    batch_iterator = tqdm(range(n_batches), desc="Computing majority distances")
else:
    batch_iterator = range(n_batches)
    print("  Progress: ", end="", flush=True)

for i in batch_iterator:
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, n_majority)
    
    # Compute distances for this batch
    from sklearn.metrics import pairwise_distances
    batch_distances = pairwise_distances(
        X_majority[start_idx:end_idx], 
        X_majority,
        metric='euclidean'
    )
    
    distances_majority[start_idx:end_idx] = batch_distances.astype(np.float32)
    
    # Print progress every 10 batches if no tqdm
    if not has_tqdm and i % 10 == 0:
        print(f"{i}/{n_batches}", end=" ", flush=True)

if not has_tqdm:
    print()  # New line after progress

elapsed = time.time() - start_time
print(f"\n  ✓ Completed in {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)")
print(f"  ✓ Distance range: [{distances_majority[np.triu_indices_from(distances_majority, k=1)].min():.3f}, {distances_majority.max():.3f}]")
print(f"  ✓ Distance mean: {distances_majority[np.triu_indices_from(distances_majority, k=1)].mean():.3f}")
print(f"  ✓ Matrix shape: {distances_majority.shape}")
print(f"  ✓ Matrix size: {distances_majority.nbytes / 1e9:.2f} GB")

# Verify symmetry
is_symmetric = np.allclose(distances_majority, distances_majority.T, rtol=1e-5)
print(f"  ✓ Matrix is symmetric: {is_symmetric}")

# Verify diagonal is zero
diagonal_max = np.abs(np.diag(distances_majority)).max()
print(f"  ✓ Diagonal max: {diagonal_max:.6f} (should be ~0)")

print("\n3. Saving majority-to-majority distances...")

# Save HDF5
print("\n  a) Saving HDF5 format...")
h5_path = f"{output_dir}/distances_majority_majority.h5"
print(f"Saving to HDF5: {h5_path}")
with h5py.File(h5_path, 'w') as f:
    f.create_dataset('distances', data=distances_majority, compression='gzip', compression_opts=4)
    f.create_dataset('majority_enrolids', data=majority_enrolids)
    f.attrs['n_samples'] = n_majority
    f.attrs['n_distances'] = total_distances
    f.attrs['computation_time_seconds'] = elapsed
    f.attrs['matrix_type'] = 'symmetric'

file_size_mb = os.path.getsize(h5_path) / 1e6
print(f"  ✓ Saved {file_size_mb:.1f} MB")


print("\n" + "="*80)
print("MAJORITY-TO-MAJORITY DISTANCES COMPLETE!")
print("="*80)
print(f"\nFiles saved:")
print(f"  - {h5_path}")
print(f"      Size: {file_size_mb:.1f} MB (compressed)")
print(f"\nComputation time: {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)")
print(f"Average time per distance: {elapsed / total_distances * 1e6:.2f} microseconds")


PRECOMPUTING MAJORITY-TO-MAJORITY DISTANCES

1. Estimating computation size...
  Majority samples: 22,819
  Total distances (full matrix): 520,706,761
  Unique distances (upper triangle): 260,341,971
  Memory (float32): 2.08 GB

  ⏱ Estimated time: 1.7 seconds (0.03 minutes)

2. Computing pairwise distances...
  Shape: (22819, 122) × (22819, 122)
  Using batched computation for memory efficiency...

  Allocated 2.08 GB for distance matrix
  Computing in 23 batches of 1000 samples...


Computing majority distances: 100%|██████████| 23/23 [00:01<00:00, 18.94it/s]



  ✓ Completed in 1.2 seconds (0.02 minutes)
  ✓ Distance range: [0.114, 192.888]
  ✓ Distance mean: 10.861
  ✓ Matrix shape: (22819, 22819)
  ✓ Matrix size: 2.08 GB
  ✓ Matrix is symmetric: True
  ✓ Diagonal max: 0.000003 (should be ~0)

3. Saving majority-to-majority distances...

  a) Saving HDF5 format...
Saving to HDF5: ./CKD_precomputed_distances_all_features/distances_majority_majority.h5
  ✓ Saved 1791.9 MB

MAJORITY-TO-MAJORITY DISTANCES COMPLETE!

Files saved:
  - ./CKD_precomputed_distances_all_features/distances_majority_majority.h5
      Size: 1791.9 MB (compressed)

Computation time: 1.2 seconds (0.02 minutes)
Average time per distance: 0.00 microseconds


In [16]:
# DIAGNOSTIC: Check if distances were actually computed
print("="*80)
print("DIAGNOSTIC: Checking computed distances")
print("="*80)

# Check if 'distances' variable exists
try:
    distances_exists = 'distances' in locals() or 'distances' in globals()
    if distances_exists:
        print("✓ distances variable exists")
        
        # Check shape
        print(f"\nDistance matrix shape: {distances.shape}")
        expected_shape = (len(majority_enrolids), len(minority_enrolids))
        print(f"Expected shape: {expected_shape}")
        
        if distances.shape == expected_shape:
            print("✓ Shape matches expected!")
        else:
            print(f"✗ Shape MISMATCH! Got {distances.shape}, expected {expected_shape}")
        
        # Check size
        n_distances = distances.shape[0] * distances.shape[1]
        print(f"\nTotal distances: {n_distances:,}")
        print(f"Expected: {len(majority_enrolids) * len(minority_enrolids):,}")
        
        # Check memory size
        memory_mb = distances.nbytes / 1e6
        print(f"\nMemory size: {memory_mb:.1f} MB")
        print(f"Data type: {distances.dtype}")
        
        # Expected size for float32: 4 bytes per number
        expected_mb = (distances.shape[0] * distances.shape[1] * 4) / 1e6
        print(f"Expected size (float32): {expected_mb:.1f} MB")
        
        # Check if distances are actually computed (not all zeros)
        print(f"\nDistance statistics:")
        print(f"  Min: {distances.min():.3f}")
        print(f"  Max: {distances.max():.3f}")
        print(f"  Mean: {distances.mean():.3f}")
        print(f"  Std: {distances.std():.3f}")
        
        if distances.min() == 0 and distances.max() == 0:
            print("  ⚠ WARNING: All distances are zero! This is wrong.")
        else:
            print("  ✓ Distances have reasonable values")
        
        # Sample a few distances
        print(f"\nSample distances (first 5x5 block):")
        print(distances[:5, :5])
        
    else:
        print("✗ distances variable does NOT exist!")
        print("  The computation may have failed silently.")
        
except Exception as e:
    print(f"✗ Error checking distances: {e}")

print("\n" + "="*80)


DIAGNOSTIC: Checking computed distances
✓ distances variable exists

Distance matrix shape: (22819, 660)
Expected shape: (22819, 660)
✓ Shape matches expected!

Total distances: 15,060,540
Expected: 15,060,540

Memory size: 60.2 MB
Data type: float32
Expected size (float32): 60.2 MB

Distance statistics:
  Min: 2.482
  Max: 192.007
  Mean: 17.035
  Std: 12.071
  ✓ Distances have reasonable values

Sample distances (first 5x5 block):
[[13.5948    11.415156  12.2083025 17.011843  12.0468025]
 [16.896803   8.957179  10.767186  15.984389   9.925173 ]
 [18.328432   9.275639  10.963341  15.1766815  9.191249 ]
 [17.499699   8.759338   9.881361  14.234861  10.031384 ]
 [17.990934   8.70764   10.154926  15.126108  10.630815 ]]

